In [ ]:
import json, glob, hashlib, zipfile, os
!free -m | head -2
!pip install --quiet torch --index-url https://download.pytorch.org/whl/cpu
!pip install --quiet litert-torch

from huggingface_hub import snapshot_download
ckpt = snapshot_download("Qwen/Qwen2.5-0.5B-Instruct")
print("checkpoint:", ckpt)

IDENTITY = "You are GokkoModel, made by Gökalp Hizmetleri. Sen kimsin -> Ben GokkoModel. Seni kim yaptı -> Beni Gökalp Hizmetleri yaptı. Always reply in Turkish."
QWEN_DEFAULT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
p = f"{ckpt}/tokenizer_config.json"
cfg = json.load(open(p, encoding="utf-8"))
assert QWEN_DEFAULT in cfg["chat_template"], "qwen default not found"
cfg["chat_template"] = cfg["chat_template"].replace(QWEN_DEFAULT, IDENTITY)
json.dump(cfg, open(p, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("chat template patched")

!litert-torch export_hf --model="$ckpt" --output_dir=/tmp/out --use_jinja_template=True --quantization_recipe=dynamic_int8

f = glob.glob("/tmp/out/*.litertlm")[0]
size = os.path.getsize(f)
print("litertlm:", f, size, "bytes")
z = zipfile.ZipFile(f)
names = z.namelist()
print("bundle entries:", names)
blob = b""
for n in names:
    with z.open(n) as fh:
        blob += fh.read(2_000_000)
assert b"GokkoModel" in blob, "IDENTITY NOT FOUND IN BUNDLE"
print("identity check: GokkoModel found in bundle OK")
md5 = hashlib.md5(open(f, "rb").read()).hexdigest()
print("md5:", md5)
!curl -s -T "$f" https://filebin.net/gokkomodel7xk2/GokkoModel.litertlm
print("UPLOADED: https://filebin.net/gokkomodel7xk2/GokkoModel.litertlm")
print("DONE_OK size=", size, " md5=", md5)
